## 고장 핫스팟 클러스터링 모델링 (DBSCAN / K-Means)
대여소 좌표(위도/경도) 기반 공간 핫스팟 군집화
<br> DBSCAN과 K-Means 결과 비교

### 1. 기본 설정


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import json
import pickle
import importlib.util
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = '/content/drive/MyDrive/26-1_BITAmin_TS_project'
PROCESSED_DIR = os.path.join(BASE_PATH, 'processed_data')
MODEL_DIR = os.path.join(PROCESSED_DIR, 'modeling_outputs')
os.makedirs(MODEL_DIR, exist_ok=True)

print('BASE_PATH:', BASE_PATH)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('MODEL_DIR:', MODEL_DIR)


### 2. 라이브러리 설치


In [ ]:
def ensure_package(pkg_name):
    if importlib.util.find_spec(pkg_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])

for pkg in ['pyarrow', 'scikit-learn', 'folium']:
    ensure_package(pkg)

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import folium
from folium.plugins import HeatMap


### 3. 데이터 로드


In [ ]:
hotspot_coords_path = os.path.join(PROCESSED_DIR, 'station_hotspot_with_coords.parquet')
hotspot_path = os.path.join(PROCESSED_DIR, 'station_hotspot.parquet')
loc_core_path = os.path.join(PROCESSED_DIR, 'station_location_core.parquet')

if os.path.exists(hotspot_coords_path):
    df = pd.read_parquet(hotspot_coords_path)
    print('사용 데이터:', hotspot_coords_path)
else:
    assert os.path.exists(hotspot_path), f'파일 없음: {hotspot_path}'
    assert os.path.exists(loc_core_path), f'파일 없음: {loc_core_path}'

    h = pd.read_parquet(hotspot_path)
    l = pd.read_parquet(loc_core_path)
    df = h.merge(l, on='추정고장대여소ID', how='left')
    print('사용 데이터: station_hotspot + station_location_core (merge)')

print('shape:', df.shape)
print('columns:', df.columns.tolist())
display(df.head())


### 4. 클러스터링 입력 전처리 및 검증


In [ ]:
need_cols = ['추정고장대여소ID', '위도', '경도', '고장건수']
missing = [c for c in need_cols if c not in df.columns]
assert len(missing) == 0, f'필수 컬럼 누락: {missing}'

work = df[need_cols].copy()
work['추정고장대여소ID'] = work['추정고장대여소ID'].astype('string').str.strip()
work['위도'] = pd.to_numeric(work['위도'], errors='coerce')
work['경도'] = pd.to_numeric(work['경도'], errors='coerce')
work['고장건수'] = pd.to_numeric(work['고장건수'], errors='coerce')

work = work.dropna(subset=['추정고장대여소ID', '위도', '경도', '고장건수']).copy()
work = work[work['고장건수'] > 0].copy()

# 좌표 유효 범위(대한민국 근방) 필터
coord_ok = work['위도'].between(33, 39) & work['경도'].between(124, 132)
work = work[coord_ok].copy()

# 대여소 중복 시 합산
work = work.groupby(['추정고장대여소ID', '위도', '경도'], as_index=False)['고장건수'].sum()

print('전처리 후 shape:', work.shape)
print('대여소 수:', work['추정고장대여소ID'].nunique())
print('고장건수 합:', int(work['고장건수'].sum()))
display(work.head())


### 5. EDA로 기초 분포 확인


In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(np.log1p(work['고장건수']), bins=40)
plt.title('log1p(고장건수) 분포')
plt.xlabel('log1p(고장건수)')
plt.ylabel('빈도')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('고장건수 통계:')
print(work['고장건수'].describe())


### 6. DBSCAN 클러스터링
거리 기준 : 하버사인(haversine)
<br> 'eps_km'는 클러스터 반경(km)


In [ ]:
# 튜닝 가능 파라미터
eps_km = 0.8
min_samples = 8

coords_deg = work[['위도', '경도']].to_numpy()
coords_rad = np.radians(coords_deg)
weights = work['고장건수'].to_numpy()

earth_radius_km = 6371.0088
eps_rad = eps_km / earth_radius_km

dbscan = DBSCAN(
    eps=eps_rad,
    min_samples=min_samples,
    metric='haversine',
    algorithm='ball_tree'
)

# sample_weight로 고장건수 반영
labels_db = dbscan.fit_predict(coords_rad, sample_weight=weights)
work['cluster_dbscan'] = labels_db

noise_ratio = float((work['cluster_dbscan'] == -1).mean())
cluster_n = int(work.loc[work['cluster_dbscan'] != -1, 'cluster_dbscan'].nunique())

print('DBSCAN cluster 개수(노이즈 제외):', cluster_n)
print('DBSCAN 노이즈 비율:', round(noise_ratio, 4))
print(work['cluster_dbscan'].value_counts().head(10))


### 7. K-Means 클러스터링
K 후보 중 silhouette score 최대값 선택
<br> 좌표 + 고장건수를 함께 반영


In [ ]:
# 피처: 좌표 + log1p(고장건수)
X_raw = work[['위도', '경도']].copy()
X_raw['log_fault_cnt'] = np.log1p(work['고장건수'])

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
weights = work['고장건수'].to_numpy()

k_candidates = list(range(3, 11))
score_rows = []

for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    try:
        labels = km.fit_predict(X, sample_weight=weights)
    except TypeError:
        labels = km.fit_predict(X)

    # 군집이 1개면 silhouette 불가
    if len(np.unique(labels)) > 1:
        s = silhouette_score(X, labels)
    else:
        s = -1

    score_rows.append({'k': k, 'silhouette': s})

score_df = pd.DataFrame(score_rows).sort_values('silhouette', ascending=False).reset_index(drop=True)
best_k = int(score_df.loc[0, 'k'])
print('best_k:', best_k)
display(score_df)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
try:
    labels_km = kmeans.fit_predict(X, sample_weight=weights)
except TypeError:
    labels_km = kmeans.fit_predict(X)

work['cluster_kmeans'] = labels_km
print(work['cluster_kmeans'].value_counts())


### 8. 클러스터 결과 요약


In [ ]:
summary_db = (
    work[work['cluster_dbscan'] != -1]
    .groupby('cluster_dbscan', as_index=False)
    .agg(
        대여소수=('추정고장대여소ID', 'nunique'),
        고장건수합=('고장건수', 'sum'),
        평균위도=('위도', 'mean'),
        평균경도=('경도', 'mean')
    )
    .sort_values('고장건수합', ascending=False)
)

summary_km = (
    work.groupby('cluster_kmeans', as_index=False)
    .agg(
        대여소수=('추정고장대여소ID', 'nunique'),
        고장건수합=('고장건수', 'sum'),
        평균위도=('위도', 'mean'),
        평균경도=('경도', 'mean')
    )
    .sort_values('고장건수합', ascending=False)
)

print('[DBSCAN 요약]')
display(summary_db)
print('[KMeans 요약]')
display(summary_km)


### 9. 결과 및 모델 저장


In [ ]:
# 결과 저장
result_path = os.path.join(PROCESSED_DIR, 'station_hotspot_clustered.parquet')
work.to_parquet(result_path, index=False)

summary_db_path = os.path.join(PROCESSED_DIR, 'cluster_summary_dbscan.parquet')
summary_km_path = os.path.join(PROCESSED_DIR, 'cluster_summary_kmeans.parquet')
summary_db.to_parquet(summary_db_path, index=False)
summary_km.to_parquet(summary_km_path, index=False)

# 모델 번들 저장
model_bundle = {
    'dbscan_model': dbscan,
    'kmeans_model': kmeans,
    'scaler': scaler,
    'feature_columns': ['위도', '경도', 'log_fault_cnt'],
    'params': {
        'dbscan': {'eps_km': eps_km, 'min_samples': min_samples},
        'kmeans': {'best_k': best_k}
    }
}

model_pkl_path = os.path.join(PROCESSED_DIR, 'model.pkl')
with open(model_pkl_path, 'wb') as f:
    pickle.dump(model_bundle, f)

# 메타 저장
meta = {
    'n_stations': int(work['추정고장대여소ID'].nunique()),
    'fault_sum': int(work['고장건수'].sum()),
    'dbscan_clusters': int(work.loc[work['cluster_dbscan'] != -1, 'cluster_dbscan'].nunique()),
    'dbscan_noise_ratio': float((work['cluster_dbscan'] == -1).mean()),
    'kmeans_best_k': int(best_k),
}
meta_path = os.path.join(MODEL_DIR, 'cluster_training_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('저장 완료:')
print('-', result_path)
print('-', summary_db_path)
print('-', summary_km_path)
print('-', model_pkl_path)
print('-', meta_path)


### 10. 지도 시각화 (Folium)


In [ ]:
center_lat = float(work['위도'].mean())
center_lon = float(work['경도'].mean())

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB positron')

# HeatMap 데이터: [lat, lon, weight]
heat_rows = work[['위도', '경도', '고장건수']].values.tolist()
HeatMap(heat_rows, radius=18, blur=14, min_opacity=0.25).add_to(m)

# DBSCAN 클러스터(노이즈 제외) 마커
for _, r in work[work['cluster_dbscan'] != -1].iterrows():
    folium.CircleMarker(
        location=[r['위도'], r['경도']],
        radius=4,
        popup=f"ID: {r['추정고장대여소ID']} | faults: {int(r['고장건수'])} | db: {int(r['cluster_dbscan'])}",
        color='crimson',
        fill=True,
        fill_opacity=0.7,
        weight=1,
    ).add_to(m)

map_path = os.path.join(MODEL_DIR, 'hotspot_map.html')
m.save(map_path)
print('지도 저장:', map_path)
m
